In [ ]:
# ============================================================
# Paper 1 — Patch Generation
# Cell 1: Imports, project paths, and frozen configuration
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\project 3\Paper1_SSL")

HARMONIZED_DIR = (
    PROJECT_ROOT /
    "02_data" /
    "harmonized_4km"
)

MASK_DIR = (
    PROJECT_ROOT /
    "02_data" /
    "masks"
)

TABLE_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "tables"
)

CV_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "spatial_cv"
)

PATCH_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "patches"
)

FIGURE_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "figures"
)

PATCH_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Frozen 14-channel order
# ------------------------------------------------------------

feature_columns = [
    "RTE_TMI",
    "CBG",
    "CBG_RES",
    "HGM",
    "K",
    "eTh",
    "eU",
    "CPD",
    "MAG_LD",
    "GRAV_LD",
    "DEM_LD",
    "ID",
    "DEM",
    "SLOPE"
]

print("Number of frozen channels:", len(feature_columns))

# ------------------------------------------------------------
# Candidate patch sizes
# ------------------------------------------------------------

PATCH_SIZES = [
    9,
    13,
    17
]

CELL_SIZE_KM = 4.0

for p in PATCH_SIZES:

    footprint_km = p * CELL_SIZE_KM
    radius_cells = (p - 1) // 2
    radius_km = radius_cells * CELL_SIZE_KM

    print(
        f"{p}x{p} patch:"
        f" footprint = {footprint_km:.0f} km,"
        f" radius = {radius_cells} cells"
        f" ({radius_km:.0f} km)"
    )

print("\nCell 1 completed.")

In [ ]:
# ============================================================
# Cell 2: Rebuild 14-channel raster cube and common mask
# ============================================================

# ------------------------------------------------------------
# Raster file inventory
# ------------------------------------------------------------

raster_files = {
    "RTE_TMI": HARMONIZED_DIR / "TMI_RTE_4km.tif",

    "CBG": HARMONIZED_DIR / "P1_CBG_4km.tif",
    "CBG_RES": HARMONIZED_DIR / "P1_CBG_Res_4km.tif",
    "HGM": HARMONIZED_DIR / "P1_HGM_4km_FINAL.tif",

    "K": HARMONIZED_DIR / "P1_K_4km.tif",
    "eTh": HARMONIZED_DIR / "P1_Th_4km.tif",
    "eU": HARMONIZED_DIR / "P1_U_4km.tif",

    "CPD": HARMONIZED_DIR / "P1_CPD_4km.tif",

    "MAG_LD": HARMONIZED_DIR / "P1_MAG_LD_4km.tif",
    "GRAV_LD": HARMONIZED_DIR / "P1_GRAV_LD_4km.tif",
    "DEM_LD": HARMONIZED_DIR / "P1_DEM_LD_4km.tif",
    "ID": HARMONIZED_DIR / "P1_ID_4km.tif",

    "DEM": HARMONIZED_DIR / "P1_DEM_4km.tif",
    "SLOPE": HARMONIZED_DIR / "P1_SLOPE_4km.tif",
}

# ------------------------------------------------------------
# Read rasters
# ------------------------------------------------------------

raster_arrays = {}

reference_shape = None
reference_transform = None
reference_crs = None

for name in feature_columns:

    path = raster_files[name]

    if not path.exists():
        raise FileNotFoundError(
            f"Missing raster: {path}"
        )

    with rasterio.open(path) as src:

        arr = src.read(1).astype("float64")

        # Convert declared NoData to NaN
        if src.nodata is not None:
            arr[np.isclose(arr, src.nodata)] = np.nan

        # Protect against inf/-inf
        arr[~np.isfinite(arr)] = np.nan

        raster_arrays[name] = arr

        # Use first raster as geometry reference
        if reference_shape is None:

            reference_shape = arr.shape
            reference_transform = src.transform
            reference_crs = src.crs

        else:

            if arr.shape != reference_shape:
                raise ValueError(
                    f"{name} has shape {arr.shape}, "
                    f"expected {reference_shape}"
                )

            if not src.transform.almost_equals(
                reference_transform
            ):
                raise ValueError(
                    f"{name} transform does not match reference."
                )

            if src.crs != reference_crs:
                raise ValueError(
                    f"{name} CRS does not match reference."
                )

# ------------------------------------------------------------
# Build cube
# ------------------------------------------------------------

cube = np.stack(
    [
        raster_arrays[name]
        for name in feature_columns
    ],
    axis=-1
)

# ------------------------------------------------------------
# Common valid-data mask
# ------------------------------------------------------------

common_mask = np.all(
    np.isfinite(cube),
    axis=-1
)

n_total = common_mask.size
n_valid = int(common_mask.sum())
n_invalid = int(n_total - n_valid)

print("Cube shape:")
print(cube.shape)

print("\nReference geometry:")
print("Rows, cols :", reference_shape)
print("CRS        :", reference_crs)
print("Transform  :", reference_transform)

print("\nCommon mask:")
print("Total cells :", n_total)
print("Valid cells :", n_valid)
print("Invalid     :", n_invalid)
print(
    f"Coverage    : "
    f"{100 * n_valid / n_total:.2f}%"
)

# ------------------------------------------------------------
# Confirm expected values from Notebook 01
# ------------------------------------------------------------

assert cube.shape == (42, 69, 14)
assert n_valid == 2718

print(
    "\nIntegrity check passed: "
    "cube and common mask match Notebook 01."
)

print("\nCell 2 completed.")

In [ ]:
# ============================================================
# Cell 3: Load spatial fold assignments and map them to raster grid
# ============================================================

# ------------------------------------------------------------
# Load the unscaled valid-cell table from Notebook 01
# ------------------------------------------------------------

valid_cells_file = (
    TABLE_DIR /
    "Paper1_14ch_valid_cells_UNSCALED.csv"
)

valid_df = pd.read_csv(valid_cells_file)

print("Valid-cell table shape:", valid_df.shape)

# ------------------------------------------------------------
# Reconstruct the same 5 west-to-east folds used in Notebook 02
# ------------------------------------------------------------

valid_df["Spatial_Fold"] = pd.qcut(
    valid_df["X"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# ------------------------------------------------------------
# Create fold grid
#
# 0 = invalid / outside common mask
# 1..5 = spatial fold assignment
# ------------------------------------------------------------

fold_grid = np.zeros(
    reference_shape,
    dtype=np.int16
)

for _, row in valid_df.iterrows():

    r = int(row["Row"])
    c = int(row["Col"])
    fold = int(row["Spatial_Fold"])

    fold_grid[r, c] = fold

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

n_fold_cells = np.count_nonzero(
    fold_grid
)

print("\nFold-grid valid cells:", n_fold_cells)
print("Expected valid cells :", n_valid)

assert n_fold_cells == n_valid

print("\nCells per fold:")

for fold in range(1, 6):

    n = int(
        np.sum(fold_grid == fold)
    )

    print(
        f"Fold {fold}: {n} cells"
    )

# ------------------------------------------------------------
# Visualize fold grid
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(11, 7)
)

masked_fold_grid = np.ma.masked_where(
    fold_grid == 0,
    fold_grid
)

im = ax.imshow(
    masked_fold_grid,
    origin="upper"
)

ax.set_title(
    "Paper 1 — Spatial Fold Assignment on 42 × 69 Raster Grid"
)

ax.set_xlabel("Column")
ax.set_ylabel("Row")

cbar = fig.colorbar(
    im,
    ax=ax,
    ticks=[1, 2, 3, 4, 5]
)

cbar.set_label("Spatial Fold")

plt.tight_layout()

fold_grid_output = (
    FIGURE_DIR /
    "Paper1_fold_assignment_raster_grid.png"
)

plt.savefig(
    fold_grid_output,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nFold-grid figure saved to:")
print(fold_grid_output)

print("\nCell 3 completed.")

In [ ]:
# ============================================================
# Cell 4: Define reusable patch-extraction utilities
# ============================================================

def extract_valid_patches(
    cube,
    common_mask,
    center_rows,
    center_cols,
    patch_size
):
    """
    Extract square multi-channel patches centered on specified
    raster cells.

    Parameters
    ----------
    cube : np.ndarray
        Raster cube with shape (rows, cols, channels).

    common_mask : np.ndarray
        Boolean mask with shape (rows, cols).
        True only where all channels are valid.

    center_rows : array-like
        Raster row indices of candidate patch centers.

    center_cols : array-like
        Raster column indices of candidate patch centers.

    patch_size : int
        Odd patch size, e.g. 9, 13, 17.

    Returns
    -------
    patches : np.ndarray
        Shape:
        (n_valid_patches, patch_size, patch_size, n_channels)

    kept_rows : np.ndarray
        Row indices of retained patch centers.

    kept_cols : np.ndarray
        Column indices of retained patch centers.

    rejected_boundary : int
        Number rejected because the patch extends beyond
        the raster boundary.

    rejected_nodata : int
        Number rejected because at least one cell inside
        the patch is outside the common valid-data mask.
    """

    # --------------------------------------------------------
    # Basic checks
    # --------------------------------------------------------

    if patch_size % 2 == 0:
        raise ValueError(
            "patch_size must be an odd integer."
        )

    n_rows, n_cols, n_channels = cube.shape

    radius = (patch_size - 1) // 2

    patches = []
    kept_rows = []
    kept_cols = []

    rejected_boundary = 0
    rejected_nodata = 0

    # --------------------------------------------------------
    # Loop through candidate patch centers
    # --------------------------------------------------------

    for r, c in zip(center_rows, center_cols):

        r = int(r)
        c = int(c)

        r0 = r - radius
        r1 = r + radius + 1

        c0 = c - radius
        c1 = c + radius + 1

        # ----------------------------------------------------
        # Reject if patch crosses raster boundary
        # ----------------------------------------------------

        if (
            r0 < 0
            or c0 < 0
            or r1 > n_rows
            or c1 > n_cols
        ):

            rejected_boundary += 1
            continue

        # ----------------------------------------------------
        # Extract local common-mask window
        # ----------------------------------------------------

        patch_mask = common_mask[
            r0:r1,
            c0:c1
        ]

        # Require every cell inside patch to be valid
        if not np.all(patch_mask):

            rejected_nodata += 1
            continue

        # ----------------------------------------------------
        # Extract 14-channel patch
        # ----------------------------------------------------

        patch = cube[
            r0:r1,
            c0:c1,
            :
        ]

        # Defensive check
        if not np.all(np.isfinite(patch)):

            rejected_nodata += 1
            continue

        patches.append(patch)
        kept_rows.append(r)
        kept_cols.append(c)

    # --------------------------------------------------------
    # Convert lists to arrays
    # --------------------------------------------------------

    if len(patches) > 0:

        patches = np.stack(
            patches,
            axis=0
        )

    else:

        patches = np.empty(
            (
                0,
                patch_size,
                patch_size,
                n_channels
            ),
            dtype=cube.dtype
        )

    kept_rows = np.array(
        kept_rows,
        dtype=int
    )

    kept_cols = np.array(
        kept_cols,
        dtype=int
    )

    return (
        patches,
        kept_rows,
        kept_cols,
        rejected_boundary,
        rejected_nodata
    )


print("Patch-extraction function defined successfully.")

print("\nCell 4 completed.")

In [ ]:
# ============================================================
# Cell 5: Test patch feasibility for 9x9, 13x13, and 17x17
# ============================================================

# Candidate centers = all cells valid in the common mask
center_rows, center_cols = np.where(common_mask)

patch_feasibility_records = []

patch_test_results = {}

for patch_size in PATCH_SIZES:

    print("=" * 70)
    print(f"TESTING PATCH SIZE: {patch_size} x {patch_size}")
    print("=" * 70)

    (
        patches,
        kept_rows,
        kept_cols,
        rejected_boundary,
        rejected_nodata
    ) = extract_valid_patches(
        cube=cube,
        common_mask=common_mask,
        center_rows=center_rows,
        center_cols=center_cols,
        patch_size=patch_size
    )

    n_candidates = len(center_rows)
    n_kept = len(kept_rows)
    n_rejected = n_candidates - n_kept

    retention_pct = (
        100.0 * n_kept / n_candidates
    )

    patch_test_results[patch_size] = {
        "patches": patches,
        "rows": kept_rows,
        "cols": kept_cols
    }

    patch_feasibility_records.append({
        "Patch_size": patch_size,
        "Candidate_centers": n_candidates,
        "Valid_patches": n_kept,
        "Rejected_boundary": rejected_boundary,
        "Rejected_NoData": rejected_nodata,
        "Total_rejected": n_rejected,
        "Retention_percent": retention_pct
    })

    print("Candidate centers :", n_candidates)
    print("Valid patches     :", n_kept)
    print("Boundary rejected :", rejected_boundary)
    print("NoData rejected   :", rejected_nodata)
    print(
        f"Retention         : "
        f"{retention_pct:.2f}%"
    )

# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------

patch_feasibility_df = pd.DataFrame(
    patch_feasibility_records
)

print("\nPATCH-SIZE FEASIBILITY SUMMARY")
print("=" * 70)

display(
    patch_feasibility_df.round(2)
)

# ------------------------------------------------------------
# Save table
# ------------------------------------------------------------

patch_feasibility_output = (
    PATCH_DIR /
    "Paper1_patch_size_feasibility.csv"
)

patch_feasibility_df.to_csv(
    patch_feasibility_output,
    index=False
)

print("\nFeasibility table saved to:")
print(patch_feasibility_output)

print("\nCell 5 completed.")

In [ ]:
# ============================================================
# Cell 6: Fold-specific patch feasibility for
#         9x9, 13x13, and 17x17 configurations
# ============================================================

def build_strict_cv_roles(
    valid_df,
    validation_fold,
    patch_size
):
    """
    Build TRAIN / BUFFER / VALIDATION center roles for a given
    patch size using the strict square-patch no-overlap rule.
    """

    radius = (patch_size - 1) // 2
    no_overlap_cells = 2 * radius

    all_rows = valid_df["Row"].to_numpy(dtype=int)
    all_cols = valid_df["Col"].to_numpy(dtype=int)
    all_folds = valid_df["Spatial_Fold"].to_numpy(dtype=int)

    val_mask = (
        all_folds == validation_fold
    )

    val_rows = all_rows[val_mask]
    val_cols = all_cols[val_mask]

    buffer_mask = np.zeros(
        len(valid_df),
        dtype=bool
    )

    candidate_indices = np.where(
        ~val_mask
    )[0]

    for idx in candidate_indices:

        row_diff = np.abs(
            val_rows - all_rows[idx]
        )

        col_diff = np.abs(
            val_cols - all_cols[idx]
        )

        overlaps_validation = np.any(
            (row_diff <= no_overlap_cells)
            &
            (col_diff <= no_overlap_cells)
        )

        if overlaps_validation:
            buffer_mask[idx] = True

    buffer_mask[val_mask] = False

    train_mask = (
        (~val_mask)
        &
        (~buffer_mask)
    )

    role = np.full(
        len(valid_df),
        "TRAIN",
        dtype=object
    )

    role[buffer_mask] = "BUFFER"
    role[val_mask] = "VALIDATION"

    return role


# ------------------------------------------------------------
# Audit every patch size and CV fold
# ------------------------------------------------------------

fold_patch_records = []

for patch_size in PATCH_SIZES:

    print("=" * 75)
    print(
        f"PATCH SIZE: {patch_size} x {patch_size}"
    )
    print("=" * 75)

    radius = (patch_size - 1) // 2
    no_overlap_km = (
        2 * radius * CELL_SIZE_KM
    )

    for fold in range(1, 6):

        # ----------------------------------------------------
        # Build patch-size-specific CV roles
        # ----------------------------------------------------

        roles = build_strict_cv_roles(
            valid_df=valid_df,
            validation_fold=fold,
            patch_size=patch_size
        )

        temp_df = valid_df.copy()
        temp_df["CV_Role"] = roles

        # ----------------------------------------------------
        # Candidate TRAIN centers
        # ----------------------------------------------------

        train_centers = temp_df[
            temp_df["CV_Role"] == "TRAIN"
        ]

        # ----------------------------------------------------
        # Candidate VALIDATION centers
        # ----------------------------------------------------

        val_centers = temp_df[
            temp_df["CV_Role"] == "VALIDATION"
        ]

        # ----------------------------------------------------
        # Extract fully valid TRAIN patches
        # ----------------------------------------------------

        (
            train_patches,
            train_rows,
            train_cols,
            train_rej_boundary,
            train_rej_nodata
        ) = extract_valid_patches(
            cube=cube,
            common_mask=common_mask,
            center_rows=train_centers["Row"].to_numpy(),
            center_cols=train_centers["Col"].to_numpy(),
            patch_size=patch_size
        )

        # ----------------------------------------------------
        # Extract fully valid VALIDATION patches
        # ----------------------------------------------------

        (
            val_patches,
            val_rows,
            val_cols,
            val_rej_boundary,
            val_rej_nodata
        ) = extract_valid_patches(
            cube=cube,
            common_mask=common_mask,
            center_rows=val_centers["Row"].to_numpy(),
            center_cols=val_centers["Col"].to_numpy(),
            patch_size=patch_size
        )

        n_train_centers = len(train_centers)
        n_val_centers = len(val_centers)

        n_train_patches = len(train_rows)
        n_val_patches = len(val_rows)

        train_retention = (
            100 * n_train_patches / n_train_centers
            if n_train_centers > 0
            else np.nan
        )

        val_retention = (
            100 * n_val_patches / n_val_centers
            if n_val_centers > 0
            else np.nan
        )

        fold_patch_records.append({
            "Patch_size": patch_size,
            "Validation_Fold": fold,
            "No_overlap_km": no_overlap_km,

            "Candidate_TRAIN_centers": n_train_centers,
            "Usable_TRAIN_patches": n_train_patches,
            "TRAIN_retention_percent": train_retention,

            "Candidate_VALIDATION_centers": n_val_centers,
            "Usable_VALIDATION_patches": n_val_patches,
            "VALIDATION_retention_percent": val_retention,

            "TRAIN_boundary_rejected": train_rej_boundary,
            "TRAIN_NoData_rejected": train_rej_nodata,

            "VAL_boundary_rejected": val_rej_boundary,
            "VAL_NoData_rejected": val_rej_nodata
        })

        print(
            f"Fold {fold}: "
            f"TRAIN={n_train_patches:4d} | "
            f"VAL={n_val_patches:4d}"
        )


# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------

fold_patch_feasibility_df = pd.DataFrame(
    fold_patch_records
)

print("\nFOLD-SPECIFIC PATCH FEASIBILITY")
print("=" * 75)

display(
    fold_patch_feasibility_df.round(2)
)


# ------------------------------------------------------------
# Compact comparison by patch size
# ------------------------------------------------------------

patch_cv_summary = (
    fold_patch_feasibility_df
    .groupby("Patch_size")
    .agg(
        min_TRAIN_patches=(
            "Usable_TRAIN_patches",
            "min"
        ),
        mean_TRAIN_patches=(
            "Usable_TRAIN_patches",
            "mean"
        ),
        min_VALIDATION_patches=(
            "Usable_VALIDATION_patches",
            "min"
        ),
        mean_VALIDATION_patches=(
            "Usable_VALIDATION_patches",
            "mean"
        ),
        mean_TRAIN_retention=(
            "TRAIN_retention_percent",
            "mean"
        ),
        mean_VALIDATION_retention=(
            "VALIDATION_retention_percent",
            "mean"
        )
    )
    .reset_index()
)

print("\nPATCH-SIZE CV SUMMARY")
print("=" * 75)

display(
    patch_cv_summary.round(2)
)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

fold_patch_output = (
    PATCH_DIR /
    "Paper1_fold_specific_patch_feasibility.csv"
)

summary_output = (
    PATCH_DIR /
    "Paper1_patch_CV_summary.csv"
)

fold_patch_feasibility_df.to_csv(
    fold_patch_output,
    index=False
)

patch_cv_summary.to_csv(
    summary_output,
    index=False
)

print("\nSaved:")
print(fold_patch_output)
print(summary_output)

print("\nCell 6 completed.")

In [ ]:
# ============================================================
# Cell 7: Freeze primary patch-size configurations
# ============================================================

PRIMARY_PATCH_SIZES = [9, 13]
EXCLUDED_PRIMARY_PATCH_SIZES = [17]

patch_decision_records = [
    {
        "Patch_size": 9,
        "Footprint_km": 36,
        "Decision": "KEEP",
        "Role": "Primary sensitivity configuration",
        "Reason": (
            "Highest spatial-CV sample retention and strong "
            "training/validation support across all folds."
        )
    },
    {
        "Patch_size": 13,
        "Footprint_km": 52,
        "Decision": "KEEP",
        "Role": "Primary candidate configuration",
        "Reason": (
            "Balances spatial context and usable sample size; "
            "52-km footprint is comparable to the retained "
            "55-km CPD spectral support."
        )
    },
    {
        "Patch_size": 17,
        "Footprint_km": 68,
        "Decision": "EXCLUDE_FROM_PRIMARY",
        "Role": "Optional supplementary sensitivity test",
        "Reason": (
            "Strict buffered spatial CV leaves too few training "
            "patches in central folds, including only 112 usable "
            "training patches in Fold 3."
        )
    }
]

patch_decision_df = pd.DataFrame(
    patch_decision_records
)

display(patch_decision_df)

decision_output = (
    PATCH_DIR /
    "Paper1_patch_size_decision.csv"
)

patch_decision_df.to_csv(
    decision_output,
    index=False
)

print("Primary patch sizes:", PRIMARY_PATCH_SIZES)
print(
    "Excluded from primary comparison:",
    EXCLUDED_PRIMARY_PATCH_SIZES
)

print("\nDecision table saved to:")
print(decision_output)

print("\nCell 7 completed.")

In [ ]:
# ============================================================
# Cell 8: Rebuild fold-specific scaled raster cubes
#         using TRAIN-derived RobustScaler parameters
# ============================================================

# ------------------------------------------------------------
# Load saved scaler parameters from Notebook 02
# ------------------------------------------------------------

scaling_params_file = (
    CV_DIR /
    "Paper1_5fold_RobustScaler_parameters.csv"
)

scaling_params_df = pd.read_csv(
    scaling_params_file
)

print("Loaded scaling parameters:")
print(scaling_params_file)

# ------------------------------------------------------------
# Create one scaled cube per fold
# ------------------------------------------------------------

scaled_cubes = {}

for fold in range(1, 6):

    print("=" * 70)
    print(f"BUILDING SCALED CUBE FOR FOLD {fold}")
    print("=" * 70)

    fold_params = (
        scaling_params_df[
            scaling_params_df["Validation_Fold"] == fold
        ]
        .set_index("Channel")
        .loc[feature_columns]
    )

    medians = fold_params[
        "Training_Median"
    ].to_numpy()

    iqrs = fold_params[
        "Training_IQR"
    ].to_numpy()

    # --------------------------------------------------------
    # Defensive check:
    # no IQR should be zero
    # --------------------------------------------------------

    if np.any(iqrs == 0):
        zero_iqr_channels = np.array(
            feature_columns
        )[iqrs == 0]

        raise ValueError(
            f"Fold {fold} contains zero-IQR channels: "
            f"{zero_iqr_channels}"
        )

    # --------------------------------------------------------
    # Apply TRAIN-derived robust scaling to entire cube
    #
    # Broadcasting:
    # cube shape      = (42, 69, 14)
    # medians shape   = (14,)
    # iqrs shape      = (14,)
    # --------------------------------------------------------

    scaled_cube = (
        cube - medians
    ) / iqrs

    # Preserve true source NoData as NaN
    scaled_cube[
        ~np.isfinite(cube)
    ] = np.nan

    scaled_cubes[fold] = scaled_cube

    # --------------------------------------------------------
    # Diagnostics
    # --------------------------------------------------------

    print("Scaled cube shape:", scaled_cube.shape)

    print(
        "Finite values:",
        np.isfinite(scaled_cube).sum()
    )

    print(
        "NaN values:",
        np.isnan(scaled_cube).sum()
    )

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

for fold in range(1, 6):

    assert scaled_cubes[fold].shape == (
        42,
        69,
        14
    )

    assert np.array_equal(
        np.isfinite(scaled_cubes[fold]),
        np.isfinite(cube)
    )

print(
    "\nIntegrity check passed: "
    "all 5 scaled cubes preserve the original valid-data mask."
)

print("\nCell 8 completed.")

In [ ]:
# ============================================================
# Cell 9: Generate and save fold-specific scaled patch tensors
#         for the retained 9x9 and 13x13 configurations
# ============================================================

PRIMARY_PATCH_SIZES = [9, 13]

patch_manifest_records = []

# ------------------------------------------------------------
# Loop over patch sizes
# ------------------------------------------------------------

for patch_size in PRIMARY_PATCH_SIZES:

    print("\n" + "=" * 80)
    print(f"PATCH CONFIGURATION: {patch_size} x {patch_size}")
    print("=" * 80)

    patch_size_dir = (
        PATCH_DIR /
        f"{patch_size}x{patch_size}"
    )

    patch_size_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Loop over spatial CV folds
    # --------------------------------------------------------

    for fold in range(1, 6):

        print("\n" + "-" * 70)
        print(
            f"Patch {patch_size}x{patch_size} | Fold {fold}"
        )
        print("-" * 70)

        # ----------------------------------------------------
        # Build the correct patch-size-specific CV roles
        # ----------------------------------------------------

        roles = build_strict_cv_roles(
            valid_df=valid_df,
            validation_fold=fold,
            patch_size=patch_size
        )

        temp_df = valid_df.copy()
        temp_df["CV_Role"] = roles

        train_centers_df = temp_df[
            temp_df["CV_Role"] == "TRAIN"
        ].copy()

        val_centers_df = temp_df[
            temp_df["CV_Role"] == "VALIDATION"
        ].copy()

        # ----------------------------------------------------
        # Use this fold's TRAIN-scaled raster cube
        # ----------------------------------------------------

        fold_cube = scaled_cubes[fold]

        # ----------------------------------------------------
        # Extract TRAIN patches
        # ----------------------------------------------------

        (
            X_train,
            train_rows,
            train_cols,
            train_boundary_rejected,
            train_nodata_rejected
        ) = extract_valid_patches(
            cube=fold_cube,
            common_mask=common_mask,
            center_rows=train_centers_df["Row"].to_numpy(),
            center_cols=train_centers_df["Col"].to_numpy(),
            patch_size=patch_size
        )

        # ----------------------------------------------------
        # Extract VALIDATION patches
        # ----------------------------------------------------

        (
            X_val,
            val_rows,
            val_cols,
            val_boundary_rejected,
            val_nodata_rejected
        ) = extract_valid_patches(
            cube=fold_cube,
            common_mask=common_mask,
            center_rows=val_centers_df["Row"].to_numpy(),
            center_cols=val_centers_df["Col"].to_numpy(),
            patch_size=patch_size
        )

        # ----------------------------------------------------
        # Convert to float32
        #
        # This cuts storage approximately in half and is the
        # standard numerical precision for neural-network work.
        # ----------------------------------------------------

        X_train = X_train.astype(
            np.float32
        )

        X_val = X_val.astype(
            np.float32
        )

        # ----------------------------------------------------
        # Build metadata for retained TRAIN centers
        # ----------------------------------------------------

        train_lookup = (
            train_centers_df
            .set_index(["Row", "Col"])
        )

        train_metadata_records = []

        for r, c in zip(
            train_rows,
            train_cols
        ):

            row_info = train_lookup.loc[
                (r, c)
            ]

            train_metadata_records.append({
                "Cell_ID": row_info["Cell_ID"],
                "Row": r,
                "Col": c,
                "X": row_info["X"],
                "Y": row_info["Y"],
                "Spatial_Fold": row_info["Spatial_Fold"],
                "Validation_Fold": fold,
                "CV_Role": "TRAIN",
                "Patch_size": patch_size
            })

        train_metadata_df = pd.DataFrame(
            train_metadata_records
        )

        # ----------------------------------------------------
        # Build metadata for retained VALIDATION centers
        # ----------------------------------------------------

        val_lookup = (
            val_centers_df
            .set_index(["Row", "Col"])
        )

        val_metadata_records = []

        for r, c in zip(
            val_rows,
            val_cols
        ):

            row_info = val_lookup.loc[
                (r, c)
            ]

            val_metadata_records.append({
                "Cell_ID": row_info["Cell_ID"],
                "Row": r,
                "Col": c,
                "X": row_info["X"],
                "Y": row_info["Y"],
                "Spatial_Fold": row_info["Spatial_Fold"],
                "Validation_Fold": fold,
                "CV_Role": "VALIDATION",
                "Patch_size": patch_size
            })

        val_metadata_df = pd.DataFrame(
            val_metadata_records
        )

        # ----------------------------------------------------
        # Fold-specific output directory
        # ----------------------------------------------------

        fold_dir = (
            patch_size_dir /
            f"Fold_{fold}"
        )

        fold_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        # ----------------------------------------------------
        # Output paths
        # ----------------------------------------------------

        train_tensor_output = (
            fold_dir /
            f"Fold{fold}_TRAIN_{patch_size}x{patch_size}_14ch.npy"
        )

        val_tensor_output = (
            fold_dir /
            f"Fold{fold}_VALIDATION_{patch_size}x{patch_size}_14ch.npy"
        )

        train_metadata_output = (
            fold_dir /
            f"Fold{fold}_TRAIN_{patch_size}x{patch_size}_metadata.csv"
        )

        val_metadata_output = (
            fold_dir /
            f"Fold{fold}_VALIDATION_{patch_size}x{patch_size}_metadata.csv"
        )

        # ----------------------------------------------------
        # Save tensors
        # ----------------------------------------------------

        np.save(
            train_tensor_output,
            X_train
        )

        np.save(
            val_tensor_output,
            X_val
        )

        # ----------------------------------------------------
        # Save patch-center metadata
        # ----------------------------------------------------

        train_metadata_df.to_csv(
            train_metadata_output,
            index=False
        )

        val_metadata_df.to_csv(
            val_metadata_output,
            index=False
        )

        # ----------------------------------------------------
        # Verify tensor geometry
        # ----------------------------------------------------

        expected_train_shape = (
            len(train_rows),
            patch_size,
            patch_size,
            14
        )

        expected_val_shape = (
            len(val_rows),
            patch_size,
            patch_size,
            14
        )

        assert X_train.shape == expected_train_shape
        assert X_val.shape == expected_val_shape

        assert len(train_metadata_df) == len(X_train)
        assert len(val_metadata_df) == len(X_val)

        assert np.all(
            np.isfinite(X_train)
        )

        assert np.all(
            np.isfinite(X_val)
        )

        # ----------------------------------------------------
        # Compare counts against Cell 6 feasibility audit
        # ----------------------------------------------------

        expected_record = (
            fold_patch_feasibility_df[
                (fold_patch_feasibility_df["Patch_size"] == patch_size)
                &
                (
                    fold_patch_feasibility_df["Validation_Fold"]
                    == fold
                )
            ]
            .iloc[0]
        )

        expected_train_n = int(
            expected_record["Usable_TRAIN_patches"]
        )

        expected_val_n = int(
            expected_record["Usable_VALIDATION_patches"]
        )

        assert len(X_train) == expected_train_n
        assert len(X_val) == expected_val_n

        # ----------------------------------------------------
        # Add manifest record
        # ----------------------------------------------------

        patch_manifest_records.append({
            "Patch_size": patch_size,
            "Validation_Fold": fold,

            "TRAIN_patches": len(X_train),
            "VALIDATION_patches": len(X_val),

            "Tensor_rows": patch_size,
            "Tensor_cols": patch_size,
            "Channels": 14,

            "TRAIN_boundary_rejected":
                train_boundary_rejected,

            "TRAIN_NoData_rejected":
                train_nodata_rejected,

            "VAL_boundary_rejected":
                val_boundary_rejected,

            "VAL_NoData_rejected":
                val_nodata_rejected,

            "TRAIN_tensor":
                str(train_tensor_output),

            "VALIDATION_tensor":
                str(val_tensor_output),

            "TRAIN_metadata":
                str(train_metadata_output),

            "VALIDATION_metadata":
                str(val_metadata_output)
        })

        print(
            "TRAIN tensor     :",
            X_train.shape
        )

        print(
            "VALIDATION tensor:",
            X_val.shape
        )

        print(
            "Data type        :",
            X_train.dtype
        )

# ------------------------------------------------------------
# Create and save manifest
# ------------------------------------------------------------

patch_manifest_df = pd.DataFrame(
    patch_manifest_records
)

patch_manifest_output = (
    PATCH_DIR /
    "Paper1_primary_patch_tensor_manifest.csv"
)

patch_manifest_df.to_csv(
    patch_manifest_output,
    index=False
)

# ------------------------------------------------------------
# Compact manifest display
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PRIMARY PATCH TENSOR MANIFEST")
print("=" * 80)

display(
    patch_manifest_df[
        [
            "Patch_size",
            "Validation_Fold",
            "TRAIN_patches",
            "VALIDATION_patches",
            "Tensor_rows",
            "Tensor_cols",
            "Channels"
        ]
    ]
)

print("\nManifest saved to:")
print(patch_manifest_output)

print(
    "\nIntegrity checks passed for all "
    "9x9 and 13x13 patch tensors."
)

print("\nCell 9 completed.")

In [ ]:
# ============================================================
# Cell 10: Visual sanity check of a real multichannel patch
# ============================================================

# ------------------------------------------------------------
# Choose a representative patch
# ------------------------------------------------------------

patch_size = 13
fold = 3
sample_index = 0

fold_dir = (
    PATCH_DIR /
    "13x13" /
    f"Fold_{fold}"
)

val_tensor_file = (
    fold_dir /
    f"Fold{fold}_VALIDATION_13x13_14ch.npy"
)

val_metadata_file = (
    fold_dir /
    f"Fold{fold}_VALIDATION_13x13_metadata.csv"
)

X_val = np.load(val_tensor_file)
val_meta = pd.read_csv(val_metadata_file)

patch = X_val[sample_index]

print("Loaded validation tensor shape:", X_val.shape)
print("Selected patch shape:", patch.shape)

print("\nSelected patch metadata:")
display(
    val_meta.iloc[[sample_index]]
)

# ------------------------------------------------------------
# Plot all 14 channels
# ------------------------------------------------------------

ncols = 4
nrows = 4

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(14, 13)
)

axes = axes.flatten()

for i, channel in enumerate(feature_columns):

    im = axes[i].imshow(
        patch[:, :, i],
        origin="upper"
    )

    axes[i].set_title(channel)
    axes[i].set_xticks([])
    axes[i].set_yticks([])

    fig.colorbar(
        im,
        ax=axes[i],
        fraction=0.046,
        pad=0.04
    )

# Remove unused panels
for j in range(len(feature_columns), len(axes)):
    fig.delaxes(axes[j])

meta_row = val_meta.iloc[sample_index]

fig.suptitle(
    f"Paper 1 — Example 13×13 Validation Patch\n"
    f"Fold {fold} | "
    f"Cell {meta_row['Cell_ID']} | "
    f"X={meta_row['X']:.0f} m, "
    f"Y={meta_row['Y']:.0f} m",
    fontsize=15
)

plt.tight_layout()

patch_fig_output = (
    FIGURE_DIR /
    "Paper1_example_13x13_multichannel_patch.png"
)

plt.savefig(
    patch_fig_output,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nFigure saved to:")
print(patch_fig_output)

# ------------------------------------------------------------
# Numerical integrity check
# ------------------------------------------------------------

print("\nPatch numerical checks")
print("-" * 60)

print(
    "All finite:",
    np.all(np.isfinite(patch))
)

print(
    "Minimum scaled value:",
    round(float(np.min(patch)), 4)
)

print(
    "Maximum scaled value:",
    round(float(np.max(patch)), 4)
)

print("\nChannel-wise patch means:")

for i, channel in enumerate(feature_columns):

    print(
        f"{channel:10s}: "
        f"{np.mean(patch[:, :, i]):8.4f}"
    )

print("\nCell 10 completed.")